In [1]:
import os
import json
import random
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import faiss
from tqdm import tqdm

from transformers import CLIPProcessor, CLIPModel

In [ ]:
from Modules.retrieval_module import Retriever
from Modules.datasets import Flickr30KImageDataset, CLIP_collate_fn
from Modules.embedding_module import Embedder

In [ ]:

TRAIN_IMAGE_DIR = "dataset/flickr30k_images/train"
TRAIN_CAPTIONS_PATH = "dataset/captions-train.csv"

TEST_IMAGE_DIR = "dataset/flickr30k_images/test"
TEST_CAPTIONS_PATH = "dataset/captions-test.csv"

FAISS_PATH = "flickr30k_clip_images.faiss"
JSON_OUT = "image_metadata.json"

TRAIN_METADATA_PATH = "train_metadata.json"
TEST_METADATA_PATH = "test_metadata.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [ ]:
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

In [ ]:
embedder = Embedder(device=DEVICE)
retriever = Retriever(JSON_OUT, FAISS_PATH)

In [ ]:
train_dataset = Flickr30KImageDataset(TRAIN_IMAGE_DIR, TRAIN_CAPTIONS_PATH, 'CLIP', CLIP_MODEL_NAME)
test_dataset  = Flickr30KImageDataset(TEST_IMAGE_DIR,  TEST_CAPTIONS_PATH, 'CLIP', CLIP_MODEL_NAME)

In [16]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=CLIP_collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=CLIP_collate_fn)

In [ ]:
loader = tqdm(train_loader, desc="Finding similar images")

for batch in loader:
    images = batch['image_tensor'].to(DEVICE) # shape [B, 3, 224, 224]
    image_names = batch['image_name']
    captions = batch['captions']
    
    embeddings = embedder.get_image_embedding(images)
    
    embeddings_np = embeddings.cpu().numpy().astype(np.float32)

    for i in range(len(images)):
        similar_images = retriever.retrieve_similar_indices(embeddings[i].unsqueeze(0), k=6)
        train_metadata[train_index] = {
            "image_name": image_names[i],
            "captions": captions[i],
            "similar_images": similar_images[1:]
        }
        train_index += 1

Finding similar images: 100%|██████████| 1924/1924 [06:54<00:00,  4.64it/s]


In [19]:
with open(TRAIN_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(train_metadata, f, indent=2, ensure_ascii=False)

In [ ]:
loader = tqdm(test_loader, desc="Finding similar images")

for batch in loader:
    images = batch['image_tensor'].to(DEVICE) # shape [B, 3, 224, 224]
    image_names = batch['image_name']
    captions = batch['captions']
    
    embeddings = embedder.get_image_embedding(images)
    
    embeddings_np = embeddings.cpu().numpy().astype(np.float32)

    for i in range(len(images)):
        similar_images = retriever.retrieve_similar_indices(embeddings[i].unsqueeze(0), k=5)
        test_metadata[test_index] = {
            "image_name": image_names[i],
            "captions": captions[i],
            "similar_images": similar_images
        }
        test_index += 1

Finding similar images: 100%|██████████| 63/63 [00:12<00:00,  4.90it/s]


In [21]:
with open(TEST_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(test_metadata, f, indent=2, ensure_ascii=False)